In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f"Dataset: {df.shape[0]} linhas x {df.shape[1]} colunas")

Dataset: 7043 linhas x 21 colunas


In [19]:
# Conhecer as colunas, tipos e nulos
for col in df.columns:
    tipo = df[col].dtype
    nulos = df[col].isnull().sum()
    unicos = df[col].nunique()
    print(f"{col:25s} | {str(tipo):8s} | Nulos: {nulos} | Únicos: {unicos}")

customerID                | object   | Nulos: 0 | Únicos: 7043
gender                    | object   | Nulos: 0 | Únicos: 2
SeniorCitizen             | int64    | Nulos: 0 | Únicos: 2
Partner                   | object   | Nulos: 0 | Únicos: 2
Dependents                | object   | Nulos: 0 | Únicos: 2
tenure                    | int64    | Nulos: 0 | Únicos: 73
PhoneService              | object   | Nulos: 0 | Únicos: 2
MultipleLines             | object   | Nulos: 0 | Únicos: 3
InternetService           | object   | Nulos: 0 | Únicos: 3
OnlineSecurity            | object   | Nulos: 0 | Únicos: 3
OnlineBackup              | object   | Nulos: 0 | Únicos: 3
DeviceProtection          | object   | Nulos: 0 | Únicos: 3
TechSupport               | object   | Nulos: 0 | Únicos: 3
StreamingTV               | object   | Nulos: 0 | Únicos: 3
StreamingMovies           | object   | Nulos: 0 | Únicos: 3
Contract                  | object   | Nulos: 0 | Únicos: 3
PaperlessBilling          | object  

## Diagnóstico Inicial

**Zero nulos** — raro, mas possível. Porém tem um problema escondido:
- `TotalCharges` está como `object` (texto) quando deveria ser numérico
- Isso significa que tem algum valor não numérico ali dentro

**Variável alvo:** `Churn` (Yes/No) — se o cliente cancelou ou não

**Colunas categóricas** (poucos valores únicos):
- Dados pessoais: gender, SeniorCitizen, Partner, Dependents
- Serviços: PhoneService, MultipleLines, InternetService, OnlineSecurity, 
  OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies
- Contrato: Contract, PaperlessBilling, PaymentMethod

**Colunas numéricas:**
- tenure (tempo como cliente em meses)
- MonthlyCharges (valor mensal)
- TotalCharges (valor total pago — precisa converter)

**customerID** é apenas identificador — não serve pra análise

In [20]:
# Encontrar os valores não numéricos em TotalCharges
# Tentar converter e ver quais falham
total_charges_numeric = pd.to_numeric(df['TotalCharges'], errors='coerce')
problemas = df[total_charges_numeric.isnull()]

print(f"Registros com TotalCharges não numérico: {len(problemas)}")
print(f"\nValores encontrados:")
print(problemas[['customerID', 'tenure', 'TotalCharges', 'MonthlyCharges']].head(15))

Registros com TotalCharges não numérico: 11

Valores encontrados:
      customerID  tenure TotalCharges  MonthlyCharges
488   4472-LVYGI       0                        52.55
753   3115-CZMZD       0                        20.25
936   5709-LVOEQ       0                        80.85
1082  4367-NUYAO       0                        25.75
1340  1371-DWPAZ       0                        56.05
3331  7644-OMVMY       0                        19.85
3826  3213-VVOLG       0                        25.35
4380  2520-SGTTA       0                        20.00
5218  2923-ARZLG       0                        19.70
6670  4075-WKNIU       0                        73.35
6754  2775-SEFEE       0                        61.90


### Decisão: TotalCharges não numérico
**Problema:** 11 registros com TotalCharges em branco (texto vazio)
**Causa:** São clientes com tenure = 0 (recém cadastrados, ainda sem pagamento)
**Decisão:** Converter TotalCharges para numérico e preencher os 11 vazios com 0,
pois o valor total pago por esses clientes é de fato zero.

In [21]:
# Converter TotalCharges para numérico e preencher vazios com 0
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

# Remover customerID (não serve pra análise)
df = df.drop('customerID', axis=1)

# Confirmar
print(f"TotalCharges tipo: {df['TotalCharges'].dtype}")
print(f"Nulos restantes: {df.isnull().sum().sum()}")
print(f"Colunas: {df.shape[1]}")

TotalCharges tipo: float64
Nulos restantes: 0
Colunas: 20


In [22]:
# Distribuição do Churn
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

print("Distribuição do Churn:")
print(f"  Não cancelou (No):  {churn_counts['No']}  ({churn_pct['No']:.1f}%)")
print(f"  Cancelou (Yes):     {churn_counts['Yes']} ({churn_pct['Yes']:.1f}%)")

Distribuição do Churn:
  Não cancelou (No):  5174  (73.5%)
  Cancelou (Yes):     1869 (26.5%)


## Análise Estatística
Objetivo: identificar quais variáveis têm relação estatisticamente 
significativa com o churn. Não basta "parecer" que influencia — 
vamos usar testes estatísticos para comprovar.

In [23]:
# Análise 1: Variáveis numéricas vs Churn
# Comparar a média de cada variável numérica entre quem cancelou e quem não cancelou

numericas = ['tenure', 'MonthlyCharges', 'TotalCharges']

for col in numericas:
    media_no = df[df['Churn'] == 'No'][col].mean()
    media_yes = df[df['Churn'] == 'Yes'][col].mean()
    
    # Teste t: verifica se a diferença entre as médias é estatisticamente significativa
    grupo_no = df[df['Churn'] == 'No'][col]
    grupo_yes = df[df['Churn'] == 'Yes'][col]
    t_stat, p_valor = stats.ttest_ind(grupo_no, grupo_yes)
    
    significativo = "SIM ✓" if p_valor < 0.05 else "NÃO ✗"
    
    print(f"\n{col}:")
    print(f"  Média (não cancelou): {media_no:.2f}")
    print(f"  Média (cancelou):     {media_yes:.2f}")
    print(f"  p-valor: {p_valor:.6f}")
    print(f"  Estatisticamente significativo? {significativo}")


tenure:
  Média (não cancelou): 37.57
  Média (cancelou):     17.98
  p-valor: 0.000000
  Estatisticamente significativo? SIM ✓

MonthlyCharges:
  Média (não cancelou): 61.27
  Média (cancelou):     74.44
  p-valor: 0.000000
  Estatisticamente significativo? SIM ✓

TotalCharges:
  Média (não cancelou): 2549.91
  Média (cancelou):     1531.80
  p-valor: 0.000000
  Estatisticamente significativo? SIM ✓


In [24]:
# Análise 2: Variáveis categóricas vs Churn
# Usar o teste Chi-quadrado: verifica se existe relação entre duas variáveis categóricas

categoricas = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
               'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
               'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
               'Contract', 'PaperlessBilling', 'PaymentMethod']

print(f"{'Variável':25s} | {'p-valor':12s} | {'Significativo?':15s} | {'Taxa churn mais alta'}")
print(f"{'-'*90}")

for col in categoricas:
    # Tabela cruzada
    tabela = pd.crosstab(df[col], df['Churn'])
    
    # Teste chi-quadrado
    chi2, p_valor, dof, expected = stats.chi2_contingency(tabela)
    
    significativo = "SIM ✓" if p_valor < 0.05 else "NÃO ✗"
    
    # Qual valor tem mais churn?
    taxas = df.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean())
    pior = taxas.idxmax()
    pior_taxa = taxas.max() * 100
    
    print(f"{col:25s} | {p_valor:12.6f} | {significativo:15s} | {pior} ({pior_taxa:.1f}%)")

Variável                  | p-valor      | Significativo?  | Taxa churn mais alta
------------------------------------------------------------------------------------------
gender                    |     0.486579 | NÃO ✗           | Female (26.9%)
SeniorCitizen             |     0.000000 | SIM ✓           | 1 (41.7%)
Partner                   |     0.000000 | SIM ✓           | No (33.0%)
Dependents                |     0.000000 | SIM ✓           | No (31.3%)
PhoneService              |     0.338783 | NÃO ✗           | Yes (26.7%)
MultipleLines             |     0.003464 | SIM ✓           | Yes (28.6%)
InternetService           |     0.000000 | SIM ✓           | Fiber optic (41.9%)
OnlineSecurity            |     0.000000 | SIM ✓           | No (41.8%)
OnlineBackup              |     0.000000 | SIM ✓           | No (39.9%)
DeviceProtection          |     0.000000 | SIM ✓           | No (39.1%)
TechSupport               |     0.000000 | SIM ✓           | No (41.6%)
StreamingTV          

### Insights — Análise Estatística

**Variáveis que NÃO influenciam o churn:**
- `gender` (p=0.49) — homens e mulheres cancelam na mesma proporção
- `PhoneService` (p=0.34) — ter telefone ou não é irrelevante

**Fatores que MAIS aumentam o risco de cancelamento:**
- **Contract Month-to-month: 42.7%** de churn — sem contrato longo, o cliente sai fácil
- **Electronic check: 45.3%** — forma de pagamento com maior churn, possível fricção
- **Fiber optic: 41.9%** — surpreendente! O serviço "premium" tem mais cancelamento.
  Pode indicar preço alto ou qualidade abaixo da expectativa
- **Sem serviços extras** (OnlineSecurity, TechSupport, DeviceProtection): ~40% de churn
  cada — clientes sem serviços adicionais são mais propensos a sair
- **SeniorCitizen: 41.7%** — idosos cancelam mais, possivelmente por dificuldade
  de uso ou sensibilidade a preço
- **Sem Partner e sem Dependents**: ~31-33% — pessoas sozinhas têm menos vínculo

**Conclusão de negócio:** O perfil de maior risco é um cliente novo, sem contrato 
longo, com fibra óptica, pagando por electronic check, sem serviços adicionais 
e morando sozinho. Esse é o cliente que a empresa precisa priorizar na retenção.

In [25]:
# Visualização: Taxa de churn pelos fatores mais relevantes
fatores_top = {
    'Contract': 'Tipo de Contrato',
    'InternetService': 'Serviço de Internet',
    'PaymentMethod': 'Forma de Pagamento',
    'TechSupport': 'Suporte Técnico'
}

fig = make_subplots(rows=2, cols=2, subplot_titles=list(fatores_top.values()))

for i, (col, titulo) in enumerate(fatores_top.items()):
    row = i // 2 + 1
    col_pos = i % 2 + 1
    
    taxas = df.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean() * 100)
    
    cores = ['#EF4444' if v > 35 else '#F59E0B' if v > 25 else '#10B981' for v in taxas.values]
    
    fig.add_trace(
        go.Bar(
            x=taxas.index,
            y=taxas.values,
            marker_color=cores,
            text=[f'{v:.1f}%' for v in taxas.values],
            textposition='outside',
            hovertemplate='%{x}<br>Churn: %{y:.1f}%<extra></extra>'
        ),
        row=row, col=col_pos
    )

fig.update_layout(
    title=dict(text='Taxa de Churn por Fator de Risco', font=dict(size=20)),
    template='plotly_white',
    showlegend=False,
    height=550
)

fig.update_yaxes(range=[0, 55])

fig.show()